# Генератор датасета продолжений на Qwen

Запускайте с GPU-рантаймом.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Сначала включите GPU-рантайм.'
print(torch.cuda.get_device_name(0))


In [ ]:
import os

try:
    import google.colab
    ENV = 'colab'
    BASE_DIR = '/content'
except ImportError:
    if os.path.exists('/kaggle'):
        ENV = 'kaggle'
        BASE_DIR = '/kaggle/working'
    else:
        ENV = 'local'
        BASE_DIR = os.getcwd()

print(f'Окружение: {ENV}, базовая директория: {BASE_DIR}')


In [ ]:
REPO_URL = 'https://github.com/Teodorus730/qwen.git'
BRANCH = 'main'
PROJECT_DIR = 'qwen_continuation_dataset'
REPO_DIR = f'{BASE_DIR}/qwen'

!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}/{PROJECT_DIR}
!python -m pip install -q -r requirements.txt


## Проверка конфигурации

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.5 \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --output outputs/mixed_entropy_20.jsonl \
  --overwrite \
  --dry-run


## Генерация — fixed-режим (FineWeb-Edu)

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset fineweb \
  --mode fixed \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --preview 3 \
  --output outputs/fineweb_fixed_20.jsonl \
  --overwrite


## Генерация — entropy-режим (FineMath)

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset math \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --preview 3 \
  --output outputs/math_entropy_20.jsonl \
  --overwrite


## Генерация — смешанные источники

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.7 \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 20 \
  --preview 3 \
  --output outputs/mixed_70_entropy_20.jsonl \
  --overwrite


## Генерация — батчинг, метаданные локально

Тот же запуск со смешанными источниками, но `--batch-size` генерирует
несколько документов за вызов модели (главный рычаг для throughput — см.
секцию Производительность в README), а `--save-extra-fields` сохраняет
счётчики токенов, энтропию и информацию об источнике в отдельный локальный
файл. Основной вывод (и всё, что уходит на Hugging Face) по-прежнему
содержит только `prefix`/`suffix`.

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.7 \
  --mode entropy \
  --batch-size 8 \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --save-extra-fields \
  --extra-fields-path outputs/mixed_batched_extra.jsonl \
  --extra-fields-batch-size 20 \
  --max-examples 40 \
  --preview 3 \
  --output outputs/mixed_batched.jsonl \
  --overwrite


## Просмотр результатов

In [ ]:
!python inspect_dataset.py \
    outputs/mixed_70_entropy_20.jsonl \
    --show-examples 3


## Детекция циклов выключена (для сравнения)

Те же настройки, что в примере с fixed-режимом выше, но с явно
отключённой детекцией циклов по n-граммам — удобно для сравнения
сырого поведения модели с выводом при включённой проверке.

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset fineweb \
  --mode fixed \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --no-cycle-detection \
  --max-examples 20 \
  --preview 3 \
  --output outputs/fineweb_fixed_no_cycle.jsonl \
  --overwrite


## Загрузка на Hugging Face

Не нужно править `config.yaml` — передайте `--hf-upload` и `--hf-repo-id`
напрямую. Авторизуйтесь ниже, затем запускайте. Шарды загружаются
инкрементально и резюмируются с последнего чекпоинта при перезапуске,
так что эту ячейку безопасно перезапускать после разрыва соединения.

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste token or run: !huggingface-cli login
HF_REPO_ID = 'your_username/qwen_continuation_dataset'


In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.5 \
  --mode entropy \
  --batch-size 8 \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --cycle-detection \
  --cycle-window-chars 100 \
  --cycle-ngram-chars 20 \
  --cycle-min-chars 50 \
  --max-examples 50000 \
  --output outputs/train.jsonl \
  --hf-upload \
  --hf-repo-id $HF_REPO_ID \
  --hf-shard-size 10000
